# 02 — Retrieval Deneyleri (İP-3)

Tüm sistemleri (5 embedding modeli + BM25) aynı soru setinde yarıştırır: Recall@5/10, nDCG@10, MRR@10, Hit@5, gecikme.

Bu notebook ince bir kabuktur; asıl mantık `scripts/deney_calistir.py` içindedir ve komut satırından da çalışır.

**Ön koşul:** `data/processed/sorular.jsonl` hazır olmalı (notebook 01).

In [1]:
# Deneyi çalıştır (GPU gerekir; modeller ilk seferde indirilir)
import os
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt
!python scripts/deney_calistir.py --config configs/deney.yaml

Cloning into '/content/RAG-proje'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 102 (delta 34), reused 86 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 93.85 KiB | 807.00 KiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/RAG-proje
From https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri
 * branch            RAG-dev    -> FETCH_HEAD
Already up to date.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Chunk sayısı: 82
Test sorusu: 120
modules.json: 100% 229/229 [00:00<00:00, 951kB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 447kB/s]
README.md: 100% 3.89k/3.89k [00:00<00:00, 13.5MB/s]
sentence

In [2]:
# En güncel özet tabloyu göster
from pathlib import Path
import pandas as pd

klasorler = sorted(Path('results').glob('ana-karsilastirma-*'))
son = klasorler[-1]
ozet = pd.read_csv(son / 'ozet.csv')
print('Deney klasörü:', son)
ozet

Deney klasörü: results/ana-karsilastirma-20261111-092927


,sistem,indeks_suresi_sn,ort_gecikme_ms,recall@5,recall@10,ndcg@10,mrr@10,hit@5
0,minilm,10.09,12.57,0.8500,0.9083,0.7477,0.6954,0.8500
1,e5-base,17.53,11.81,0.9750,0.9917,0.9246,0.9031,0.9750
2,bge-m3,60.86,20.46,0.9833,0.9917,0.9189,0.8943,0.9833
3,qwen3-0.6b,36.80,62.74,0.9083,0.9417,0.8165,0.7751,0.9083
4,tr-model,10.38,15.76,0.9333,0.9583,0.8336,0.7924,0.9333
5,bm25,0.01,0.41,0.9250,0.9417,0.8479,0.8165,0.9250


In [3]:
# Eşleştirilmiş bootstrap: en iyi dense model vs BM25 (nDCG@10) + Soru tipi kırılımı (RQ2)
import sys
sys.path.insert(0, '/content/RAG-proje')
import numpy as np
from src.metrics import eslestirilmis_bootstrap_fark

soru = pd.read_csv(son / 'soru-bazinda.csv')
metrik = soru[soru['metrik'] == 'ndcg@10']
pivot = metrik.pivot(index='soru_id', columns='sistem', values='skor')

en_iyi_dense = ozet[ozet['sistem'] != 'bm25'].sort_values('ndcg@10', ascending=False).iloc[0]['sistem']
a = pivot[en_iyi_dense].to_numpy()
b = pivot['bm25'].to_numpy()
sonuc = eslestirilmis_bootstrap_fark(a, b, tekrar=1000)
print(f'{en_iyi_dense} - bm25 | fark: {sonuc["fark"]:.4f} '
      f'GA: [{sonuc["alt"]:.4f}, {sonuc["ust"]:.4f}] anlamlı: {sonuc["anlamli"]}')

print('\n--- Soru Tipine Göre nDCG@10 Kırılımı (RQ2) ---')
kirilim = metrik.groupby(['sistem', 'tip'])['skor'].mean().unstack().round(4)
print(kirilim)
print('\nSonuç klasörünü Drive\'a yedeklemeyi ve bu notebook\'u GitHub\'a kaydetmeyi unutmayın.')

e5-base - bm25 | fark: 0.0767 GA: [0.0269, 0.1283] anlamlı: True

--- Soru Tipine Göre nDCG@10 Kırılımı (RQ2) ---
tip         bicimbilimsel  cok-maddeli  olgusal  paraphrase
sistem                                                     
bge-m3             0.9441       0.9254   0.8841      0.9218
bm25               0.8841       0.9462   0.9206      0.6407
e5-base            0.9350       0.9544   0.9249      0.8842
minilm             0.8256       0.7936   0.7186      0.6531
qwen3-0.6b         0.8632       0.8714   0.8121      0.7193
tr-model           0.8585       0.8977   0.8188      0.7592

Sonuç klasörünü Drive'a yedeklemeyi ve bu notebook'u GitHub'a kaydetmeyi unutmayın.


In [ ]:
# A4 (Top-k = 1,3,5,10), A5 (Chunk Boyutu = 400/50, 800/100, 1200/150) ve Standart Türkçe Benchmark Deneyleri
import os
REPO_URL = "https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git"
BRANCH = "RAG-dev"
REPO_DIR = "/content/RAG-proje"
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt
!python scripts/ablasyon_calistir.py --config configs/deney.yaml

import pandas as pd
print("\n=== A4: Top-k Erişim Ablasyonu ===")
print(pd.read_csv("results/a4-topk-ablasyon.csv").to_string(index=False))

print("\n=== A5: Chunk Uzunluğu ve İndeks Boyutu (KB) Ablasyonu ===")
print(pd.read_csv("results/a5-chunk-ablasyon.csv").to_string(index=False))

print("\n=== Bölüm 3: Bağımsız Standart Türkçe Benchmark Karşılaştırması ===")
print(pd.read_csv("results/dis-benchmark-ozet.csv").to_string(index=False))

## Yorum rehberi

- Güven aralığı 0'ı içermiyorsa fark istatistiksel olarak anlamlıdır (H1 için kanıt).
- Soru tipine göre kırılım için `soru-bazinda.csv` içindeki `tip` sütununu kullanın (RQ2).
- TR-MTEB/TR-TEB standart görevleri ayrı çalışmalardır; `datasets` ile indirilip aynı metrik koduyla değerlendirilir.